# Common · 00 — Configuration, paths and utilities

Loaded at the top of every phase notebook with `%run common/00_config.ipynb`.
This is the **single source of truth** for every experiment: dataset version, preprocessing,
the discrete RL action space, the reward weights, and deep-model hyper-parameters.

In [ ]:
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

ROOT = Path.cwd()
while not (ROOT / "notebooks").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CFG = yaml.safe_load(r"""
seed: 42

paths:
  raw: data/raw/circor_v1.0.3
  metadata: data/interim/metadata
  segments: data/interim/segments
  processed: data/processed
  models: models
  results: results

data:
  dataset_version: "1.0.3"
  base_url: https://physionet.org/files/circor-heart-sound/1.0.3/
  split: {train: 0.70, val: 0.15, test: 0.15}
  exclude_unknown_murmur: true

preprocessing:
  fs_in: 4000
  fs: 2000
  band: [20, 800]
  filter_order: 4
  spike_removal: true
  trim_to_annotation: true
  min_duration_s: 4.0

qc:
  clip_fraction_max: 0.01
  silence_rms_min: 1.0e-4
  snr_db_min: 3.0

actions:
  wavelet: [db4, db6, db8, sym4, coif3]
  level: [3, 4, 5]
  band: [[20, 100], [100, 200], [200, 400], [400, 800], [20, 800]]
  window_s: [0.5, 1.0, 2.0, 4.0]
  fusion: [T, S, W, TS, TW, SW, TSW]

baseline_config:
  wavelet: db6
  level: 5
  band: [20, 800]
  window_s: 2.0
  fusion: TSW

reward:
  weights: {balanced: 0.40, sensitivity: 0.25, specificity: 0.20, robustness: 0.10, complexity: 0.05}
  robustness_snr_db: 10.0
  robustness_noise: pink
  delta_vs_baseline: true
  oracle: deep

context:
  n_contexts: 6
  n_pca: 6

rl:
  episodes: 60000
  alpha: 0.1
  epsilon: {start: 1.0, end: 0.05, decay_frac: 0.7}
  n_step: 3
  dyna_planning_steps: 10
  seeds: [0, 1, 2]

deep:
  n_fft: 128
  hop: 32
  spec_rows: 64
  wav_rows: 32
  tf_frames: 128
  epochs: 30
  crops_per_record: 4
  batch_size: 64
  lr: 2.0e-3
  weight_decay: 1.0e-4
  focal_gamma: 2.0
  patience: 8
  seeds: [0, 1, 2]
""")

P = {}
for k, v in CFG["paths"].items():
    P[k] = ROOT / v
    P[k].mkdir(parents=True, exist_ok=True)
P["figures"], P["tables"] = P["results"] / "figures", P["results"] / "tables"


P["half"] = "shared"


def use_half(name: str, oracle: str | None = None) -> None:
    P["half"] = name
    P["results"] = ROOT / "results" / name
    for k in ("figures", "tables"):
        P[k] = P["results"] / k
        P[k].mkdir(parents=True, exist_ok=True)
    if oracle is not None:
        CFG["reward"]["oracle"] = oracle
    print(f"{name}: results -> results/{name}/ | reward oracle: {CFG['reward']['oracle']}")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def save_json(obj, path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))


def load_json(path):
    with open(path) as f:
        return json.load(f)


def savefig(fig, name: str) -> None:
    fig.savefig(P["figures"] / f"{name}.png", dpi=150, bbox_inches="tight")


pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print(f"CirCor version {CFG['data']['dataset_version']}")